# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [ ]:
import pandas as pd
import scipy.stats as st
import numpy as np

# load data
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")

# Filter HP values for Dragon and Grass types
dragon_hp = df[df["Type 1"] == "Dragon"]["HP"]
grass_hp = df[df["Type 1"] == "Grass"]["HP"]

# Quick look at sample sizes and means
print("Dragon: n =", len(dragon_hp), ", mean HP =", dragon_hp.mean())
print("Grass:  n =", len(grass_hp), ", mean HP =", grass_hp.mean())

# We use an independent two-sample t-test (one-sided), since we're comparing
# the means of two independent, unrelated groups (Dragon vs Grass Pokemon).
# H0: mean(Dragon HP) <= mean(Grass HP)
# H1: mean(Dragon HP) >  mean(Grass HP)

t_stat, p_value_two_sided = st.ttest_ind(dragon_hp, grass_hp, equal_var=False)  # Welch's t-test

# Convert to one-sided p-value (since our hypothesis is directional: Dragon > Grass)
if t_stat > 0:
    p_value_one_sided = p_value_two_sided / 2
else:
    p_value_one_sided = 1 - (p_value_two_sided / 2)

print("\nt-statistic:", t_stat)
print("one-sided p-value:", p_value_one_sided)

alpha = 0.05
if p_value_one_sided < alpha:
    print("\nReject H0: Dragon Pokemon have significantly higher HP than Grass Pokemon.")
else:
    print("\nFail to reject H0: No significant evidence that Dragon Pokemon have higher HP than Grass Pokemon.")

- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [ ]:
import pandas as pd
import scipy.stats as st
import numpy as np

# load data
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")

# Stats we want to compare
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

# Split data into Legendary and Non-Legendary groups
legendary = df[df["Legendary"] == True]
non_legendary = df[df["Legendary"] == False]

print("Legendary: n =", len(legendary))
print("Non-Legendary: n =", len(non_legendary))
print("-" * 60)

alpha = 0.05
results = []

for stat in stats:
    leg_values = legendary[stat]
    non_leg_values = non_legendary[stat]

    # Independent two-sample t-test (two-sided), since we're testing
    # whether the means are simply DIFFERENT (not directionally higher/lower)
    # H0: mean(Legendary) == mean(Non-Legendary)
    # H1: mean(Legendary) != mean(Non-Legendary)
    t_stat, p_value = st.ttest_ind(leg_values, non_leg_values, equal_var=False)  # Welch's t-test

    significant = p_value < alpha

    results.append({
        "Stat": stat,
        "Legendary Mean": round(leg_values.mean(), 2),
        "Non-Legendary Mean": round(non_leg_values.mean(), 2),
        "t-statistic": round(t_stat, 3),
        "p-value": round(p_value, 6),
        "Significant (p<0.05)": significant
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [ ]:
import pandas as pd
import scipy.stats as st
import numpy as np

# load data
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")

# --- Step 1: function to compute euclidean distance ---
def euclidean_distance(lon1, lat1, lon2, lat2):
    return np.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)

# School and Hospital coordinates
school_coords = (-118, 34)
hospital_coords = (-122, 37)

# --- Step 2: calculate distances from each house to school and hospital ---
df["distance_to_school"] = euclidean_distance(
    df["longitude"], df["latitude"], school_coords[0], school_coords[1]
)

df["distance_to_hospital"] = euclidean_distance(
    df["longitude"], df["latitude"], hospital_coords[0], hospital_coords[1]
)

# --- Step 3: classify houses as "close" if within 0.50 of EITHER school or hospital ---
df["close_to_amenity"] = (df["distance_to_school"] < 0.50) | (df["distance_to_hospital"] < 0.50)

# Split into two groups
close_houses = df[df["close_to_amenity"] == True]["median_house_value"]
far_houses = df[df["close_to_amenity"] == False]["median_house_value"]

print("Close to school/hospital: n =", len(close_houses), ", mean value =", close_houses.mean())
print("Far from school/hospital: n =", len(far_houses), ", mean value =", far_houses.mean())
print("-" * 60)

# --- Step 4: choose and run the proper test ---
# Independent two-sample t-test (one-sided), since we're comparing means of two
# independent groups (close vs far), and the hypothesis is directional:
# H0: mean(close) <= mean(far)
# H1: mean(close) >  mean(far)

t_stat, p_value_two_sided = st.ttest_ind(close_houses, far_houses, equal_var=False)  # Welch's t-test

# Convert to one-sided p-value (directional hypothesis: close > far)
if t_stat > 0:
    p_value_one_sided = p_value_two_sided / 2
else:
    p_value_one_sided = 1 - (p_value_two_sided / 2)

print("t-statistic:", t_stat)
print("one-sided p-value:", p_value_one_sided)

alpha = 0.05
if p_value_one_sided < alpha:
    print("\nReject H0: Houses close to a school or hospital are significantly more expensive.")
else:
    print("\nFail to reject H0: No significant evidence that proximity to a school or hospital increases house value.")

In [ ]:
If p-value < 0.05: reject H0 — there's statistically significant evidence that houses close to a school or hospital have higher median house values.
If p-value ≥ 0.05: fail to reject H0 — insufficient evidence to support the claim.

When you run this, you'll likely find a statistically significant result: the "close" group's mean tends to be a fair bit higher, largely because both coordinate pairs land near expensive coastal/urban areas (near Los Angeles and the San Francisco Bay Area) — so the effect may be capturing "expensive urban region" as much as "distance to school/hospital" specifically. Worth keeping in mind as a caveat when interpreting causality. Run the code on the full dataset to see the exact numbers.